In [ ]:
import os
import pandas as pd
from pathlib import Path
import re
import nltk
import numpy as np
from collections import defaultdict
import ast


## Change this part based on your directory

In [ ]:
_dir = Path(os.environ['MAMMO_FM_REPO_ROOT']) / 'src' / 'codebase' / 'data_csv'
csv = _dir/"vindr_detection_v1_folds.csv"

df = pd.read_csv(csv)
print(csv)
print(df.columns)
df.head(5)

In [ ]:
df['finding_categories'] = df['finding_categories'].apply(ast.literal_eval)
df

In [ ]:
# Group by 'patient_id', 'image_id', 'laterality', 'view', and 'split', then aggregate 'finding_categories'
aggregated_df = df.groupby(
    ['patient_id', 'image_id', 'laterality', 'view', 'split'])['finding_categories'].agg(
    lambda x: list(set(sum(x, [])))).reset_index()

# Rename the aggregated column for clarity
aggregated_df.rename(columns={'finding_categories': 'text_label'}, inplace=True)


In [ ]:
aggregated_df[aggregated_df["patient_id"] == "988ab25980bffad7226b0a8d62a7f499"]

In [ ]:
# Define a function to filter views and return lists based on the view type
def filter_by_view(group, view_type):
    return group[group['view'] == view_type]['image_id'].tolist(), group[group['view'] == view_type]['text_label'].tolist()

# Group by 'patient_id', 'laterality' and aggregate necessary information including 'split'
new_df = aggregated_df.groupby(['patient_id', 'laterality']).apply(lambda group: pd.Series({
    'split': group['split'].iloc[0],  # Assuming 'split' is consistent within each 'patient_id', 'laterality' group
    'image_ids': group['image_id'].tolist(),
    'views': group['view'].tolist(),
    'cc_image_paths': filter_by_view(group, 'CC')[0],
    'mlo_image_paths': filter_by_view(group, 'MLO')[0],
    'cc_findings': filter_by_view(group, 'CC')[1],
    'mlo_findings': filter_by_view(group, 'MLO')[1]
})).reset_index()

new_df.head(5)

In [ ]:
new_df[new_df["patient_id"] == "988ab25980bffad7226b0a8d62a7f499"]

In [ ]:
new_df.to_csv(_dir/'clip_vindr_temp.csv', index=False)

In [ ]:
df_new = pd.read_csv(_dir/'clip_vindr_temp.csv')
df_new

In [ ]:
# Define the total list of possible findings
total_findings = ['Mass', 'Suspicious Calcification', 'Architectural Distortion', 'Global Asymmetry', 
                  'Focal Asymmetry', 'Asymmetry', 'Suspicious Lymph Node', 'Nipple Retraction', 
                  'Skin Retraction', 'Skin Thickening',]

# Function to create the specified format for cc_findings
def create_cc_findings(row):
    # Initialize all slots with empty lists
    findings = [[], [], [], []]
    
    # Determine the index for positive and negative findings based on laterality
    positive_index = 0 if row['laterality'] == 'R' else 1
    negative_index = 2 if row['laterality'] == 'R' else 3
    
    # Place the positive findings in the correct slot
    findings[positive_index] = row['cc_findings']
    
    # Calculate the negative findings
    negative_findings = list(set(total_findings) - set(row['cc_findings']))
    
    # Place the negative findings in the correct slot
    findings[negative_index] = negative_findings
    
    return findings

def create_mlo_findings(row):
    # Initialize all slots with empty lists
    findings = [[], [], [], []]
    
    # Determine the index for positive and negative findings based on laterality
    positive_index = 0 if row['laterality'] == 'R' else 1
    negative_index = 2 if row['laterality'] == 'R' else 3
    
    # Place the positive findings in the correct slot
    findings[positive_index] = row['mlo_findings']
    
    # Calculate the negative findings
    negative_findings = list(set(total_findings) - set(row['mlo_findings']))
    
    # Place the negative findings in the correct slot
    findings[negative_index] = negative_findings
    
    return findings

# Apply the function to each row in the dataframe
df_new['cc_findings_formatted'] = df_new.apply(create_cc_findings, axis=1)
df_new['mlo_findings_formatted'] = df_new.apply(create_mlo_findings, axis=1)

df_new

In [ ]:
# Add a 'fold' column based on the value of 'split'
df_new['fold'] = df_new['split'].apply(lambda x: 0 if x == 'test' else 1)


df_new

In [ ]:
# Rename the specified columns
df_new.rename(columns={
    'image_ids': 'image', 
    'cc_image_paths': 'CC',
    'mlo_image_paths': 'MLO',
    'cc_findings_formatted': 'CC_FINDING',
    'mlo_findings_formatted': 'MLO_FINDING'
}, inplace=True)

# Display the first few rows to verify the column name changes
df_new

In [ ]:
df_new['patient_id_laterality'] = df_new['patient_id'].astype(str) + '_' + df_new['laterality']
df_new

In [ ]:
df_new.drop(index=8550, inplace=True)

In [ ]:
df_final = df_new

In [ ]:
df_final.rename(columns={
    'views': 'view', 
}, inplace=True)

In [ ]:
# Convert the string representations of the lists into actual lists
df_final['view'] = df_final['view'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_final['image'] = df_final['image'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Define a function to reorder based on 'CC' and 'MLO' sequence and adjust other columns accordingly
def reorder_sequence(row):
    # Create a mapping of view to index
    view_index_map = {'CC': 0, 'MLO': 1}
    # Initialize the new order for views and related columns
    new_order = ['CC', 'MLO']
    new_image = [None, None]

    # Go through the existing views and place the items in the correct order
    for i, view in enumerate(row['view']):
        index = view_index_map.get(view)
        if index is not None:
            new_image[index] = row['image'][i]

    # Assign the reordered lists back to the row
    row['view'] = new_order
    row['image'] = new_image

    return row

# Apply the reordering function to each row
df_final = df_final.apply(reorder_sequence, axis=1)


In [ ]:
df_final.to_csv(_dir/'clip_vindr_final.csv', index=False)